# Mortgage Default API - Test Notebook

Tests the FastAPI serving layer for the mortgage default model.

**Before running this notebook**, make sure the Docker container is running:
```bash
docker build -t mortgage-default-api .
docker run -p 8000:8000 -d mortgage-default-api
```

Then open the interactive API docs at: http://localhost:8000/docs

**What the API does**: receives raw loan origination features (same format
as the Freddie Mac origination file), runs them through the same cleaning
and feature engineering pipeline used during training, and returns a
default risk prediction.

In [1]:
import json
import requests
import pandas as pd

BASE_URL = "http://127.0.0.1:8000"

def pretty(response):
    print("Status:", response.status_code)
    try:
        print(json.dumps(response.json(), indent=2))
    except Exception:
        print(response.text)

## 1. Health check

In [2]:
response = requests.get(f"{BASE_URL}/health")
pretty(response)
assert response.status_code == 200

Status: 200
{
  "model_ready": true,
  "model_uri": "data/06_models/mlflow_model",
  "metadata_path": "data/08_reporting/model_training_metadata.json",
  "feature_transformers_path": "data/04_feature/feature_transformers.pkl",
  "error": null,
  "status": "ok"
}


## 2. Model metadata

Shows which model family was selected, threshold, and number of features.

In [3]:
response = requests.get(f"{BASE_URL}/metadata")
pretty(response)
assert response.status_code == 200

Status: 200
{
  "model_uri": "data/06_models/mlflow_model",
  "selected_model_family": "gradient_boosting",
  "threshold": 0.1061764450809418,
  "target_column": "default",
  "id_column": "loan_sequence_number",
  "n_features": 24
}


## 3. Define test loans

Three synthetic loans representing different risk profiles, using 2007
origination dates - the crisis period the model was never trained on.

- **Loan 1**: low-risk profile (high credit score, low LTV, conservative DTI)
- **Loan 2**: high-risk profile (low credit score, high LTV, high DTI) 
  typical of the riskier loans that proliferated before the 2008 crisis
- **Loan 3**: very low-risk profile (excellent credit, low leverage)

In [ ]:
# Low-risk loan - originated 2007, conservative profile
loan_1 = {
    "loan_sequence_number": "TEST_001",
    "credit_score": 720,
    "first_payment_date": 200703,   # 2007
    "first_time_homebuyer_flag": "N",
    "maturity_date": 203702,         # 30-year mortgage
    "mi_percentage": 0,
    "number_of_units": 1,
    "occupancy_status": "P",
    "original_cltv": 80,
    "original_dti": 35,
    "original_upb": 200000,
    "original_ltv": 80,
    "original_interest_rate": 6.5,
    "channel": "R",
    "prepayment_penalty_flag": "N",
    "amortization_type": "FRM",
    "interest_only_indicator": "N",
    "property_state": "CA",
    "property_type": "SF",
    "postal_code": "90000",
    "loan_purpose": "P",
    "original_loan_term": 360,
    "number_of_borrowers": 2,
    "seller_name": "OTHER",
    "servicer_name": "OTHER",
    "super_conforming_flag": "N",
}

# High-risk loan - low credit score, high LTV/DTI, Florida (epicentro da crise)
loan_2 = {
    "loan_sequence_number": "TEST_002",
    "credit_score": 580,
    "first_payment_date": 200703,
    "first_time_homebuyer_flag": "Y",
    "maturity_date": 203702,
    "mi_percentage": 25,
    "number_of_units": 1,
    "occupancy_status": "P",
    "original_cltv": 97,
    "original_dti": 52,
    "original_upb": 320000,
    "original_ltv": 97,
    "original_interest_rate": 8.5,
    "channel": "B",
    "prepayment_penalty_flag": "N",
    "amortization_type": "FRM",
    "interest_only_indicator": "N",
    "property_state": "FL",
    "property_type": "SF",
    "postal_code": "33000",
    "loan_purpose": "C",
    "original_loan_term": 360,
    "number_of_borrowers": 1,
    "seller_name": "OTHER",
    "servicer_name": "OTHER",
    "super_conforming_flag": "N",
}

# Very low-risk loan — excellent credit, low leverage
loan_3 = loan_1.copy()
loan_3.update({
    "loan_sequence_number": "TEST_003",
    "credit_score": 800,
    "original_dti": 22,
    "original_ltv": 60,
    "original_cltv": 60,
    "original_interest_rate": 6.0,
    "original_upb": 150000,
    "property_state": "TX",
    "loan_purpose": "N",
})

print("Loans defined:")
print(f"  Loan 1 - credit_score={loan_1['credit_score']}, ltv={loan_1['original_ltv']}, dti={loan_1['original_dti']}")
print(f"  Loan 2 - credit_score={loan_2['credit_score']}, ltv={loan_2['original_ltv']}, dti={loan_2['original_dti']} (HIGH RISK)")
print(f"  Loan 3 - credit_score={loan_3['credit_score']}, ltv={loan_3['original_ltv']}, dti={loan_3['original_dti']}")

Loans defined:
  Loan 1 — credit_score=720, ltv=80, dti=35
  Loan 2 — credit_score=580, ltv=97, dti=52 (HIGH RISK)
  Loan 3 — credit_score=800, ltv=60, dti=22


## 4. Predict one loan (default threshold)

In [5]:
response = requests.post(f"{BASE_URL}/predict-one", json={"features": loan_1})
pretty(response)
assert response.status_code == 200

Status: 200
{
  "loan_sequence_number": "TEST_001",
  "prediction": 0,
  "probability_default": 0.0015100031978430468,
  "threshold": 0.1061764450809418,
  "model_uri": "data/06_models/mlflow_model"
}


## 5. Predict one loan (custom threshold)

Overriding the trained threshold  useful to test sensitivity.

In [6]:
response = requests.post(
    f"{BASE_URL}/predict-one",
    json={"features": loan_1, "threshold": 0.05}
)
pretty(response)
assert response.status_code == 200

Status: 200
{
  "loan_sequence_number": "TEST_001",
  "prediction": 0,
  "probability_default": 0.0015100031978430468,
  "threshold": 0.05,
  "model_uri": "data/06_models/mlflow_model"
}


## 6. Batch prediction - 3 loans

The /predict endpoint scores multiple loans in a single call.

In [7]:
batch_payload = {
    "rows": [loan_1, loan_2, loan_3]
}

response = requests.post(f"{BASE_URL}/predict", json=batch_payload)
pretty(response)
assert response.status_code == 200

Status: 200
{
  "n_rows": 3,
  "predictions": [
    {
      "loan_sequence_number": "TEST_001",
      "prediction": 0,
      "probability_default": 0.0015100031978430468,
      "threshold": 0.1061764450809418,
      "model_uri": "data/06_models/mlflow_model"
    },
    {
      "loan_sequence_number": "TEST_002",
      "prediction": 1,
      "probability_default": 0.26333329585439175,
      "threshold": 0.1061764450809418,
      "model_uri": "data/06_models/mlflow_model"
    },
    {
      "loan_sequence_number": "TEST_003",
      "prediction": 0,
      "probability_default": 0.0055320866638183235,
      "threshold": 0.1061764450809418,
      "model_uri": "data/06_models/mlflow_model"
    }
  ]
}


In [8]:
# Show predictions as a DataFrame for easy comparison
batch_result = response.json()
predictions_df = pd.DataFrame(batch_result["predictions"])

# Add risk profile labels
predictions_df['risk_profile'] = ['Low risk', 'High risk', 'Very low risk']
predictions_df[['loan_sequence_number', 'risk_profile', 'prediction',
                'probability_default', 'threshold']]

,loan_sequence_number,risk_profile,prediction,probability_default,threshold
0,TEST_001,Low risk,0,0.001510,0.106176
1,TEST_002,High risk,1,0.263333,0.106176
2,TEST_003,Very low risk,0,0.005532,0.106176


## 7. Reload model endpoint

Forces the service to reload the model from disk useful after retraining.

In [9]:
response = requests.post(f"{BASE_URL}/reload-model")
pretty(response)
assert response.status_code == 200

Status: 200
{
  "model_ready": true,
  "model_uri": "data/06_models/mlflow_model",
  "metadata_path": "data/08_reporting/model_training_metadata.json",
  "feature_transformers_path": "data/04_feature/feature_transformers.pkl",
  "error": null,
  "status": "ok"
}


## 8. Full test:  all endpoints

In [11]:
checks = {}

checks["health"] = requests.get(f"{BASE_URL}/health").status_code
checks["metadata"] = requests.get(f"{BASE_URL}/metadata").status_code
checks["predict_one"] = requests.post(
    f"{BASE_URL}/predict-one",
    json={"features": loan_1},
).status_code
checks["predict_batch"] = requests.post(
    f"{BASE_URL}/predict",
    json={"rows": [loan_1, loan_2, loan_3]},
).status_code
checks["reload_model"] = requests.post(f"{BASE_URL}/reload-model").status_code

print("=== Endpoint status codes ===")
for endpoint, status in checks.items():
    print(f"{endpoint}: {status}")

assert all(s == 200 for s in checks.values()), "One or more endpoints failed!"
print("\nAll endpoints OK!")

=== Endpoint status codes ===
health: 200
metadata: 200
predict_one: 200
predict_batch: 200
reload_model: 200

All endpoints OK!
